# Homework 07 - Classification Boundary

目标：从回归过渡到分类。你要写出 `score -> label`、`margin = y*score`、`hinge_loss`，最后生成红点蓝点和边界线。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True


def check_or_todo(condition, message):
    if not condition:
        print(message)
        return False
    return True

## Modify - 改安全距离

标准 hinge loss 用 `1` 当安全距离。如果改成 `2`，就是要求点离边界更远。

In [ ]:
# y=1, score=1.5。如果安全距离是 2，loss = relu(2 - y*score)
student_hinge_margin2 = None


def qa_check_07_modify():
    if not todo_guard([student_hinge_margin2]): return
    assert near(student_hinge_margin2, 0.5)
    print('OK: Modify 通过。')

qa_check_07_modify()

## 完整例子 - score 的符号决定类别

In [ ]:
def example_label_from_score(score):
    return 1 if score > 0 else -1

for s in [-2.0, -0.1, 0.3, 4.0]:
    print(s, '->', example_label_from_score(s))

## 作业 1-3 - label / margin / hinge loss

In [ ]:
from micrograd.engine import Value


def label_from_score(score):
    # TODO 1: score > 0 返回 1，否则返回 -1
    pass


def margin(score, y):
    # TODO 2: 返回 y * score，让答对方向统一变成正数
    pass


def hinge_loss(score, y):
    # TODO 3: 返回 relu(1 - margin)
    pass


def qa_check_07_hinge():
    try:
        assert label_from_score(0.3) == 1
        assert label_from_score(-0.1) == -1
        assert margin(Value(2.0), 1).data == 2.0
        assert margin(Value(-2.0), -1).data == 2.0
        assert hinge_loss(Value(2.0), 1).data == 0.0
        assert near(hinge_loss(Value(0.2), 1).data, 0.8)
        assert near(hinge_loss(Value(-1.0), 1).data, 2.0)
    except Exception as exc:
        print('请继续完成 label_from_score / margin / hinge_loss。当前:', type(exc).__name__, exc)
        return
    print('OK: 作业 1-3 分类 loss 通过。')

qa_check_07_hinge()

<details><summary>Show / Hide 提示</summary>

`y` 只取 `1` 或 `-1`。如果答对，`y*score` 会是正数；如果答错，会是负数。`hinge_loss = relu(1 - y*score)` 惩罚答错和不够远的答对。

</details>

## 作业 4 - 画数据和边界，不依赖 matplotlib

这里用标准库直接生成 SVG。红点是 `1`，蓝点是 `-1`，黑线是 `score=0` 的边界。

In [ ]:
xs = [(-2,-1), (-1,-2), (-2,1), (-1,1), (1,2), (2,1), (1,-1), (2,-2), (0.5,1.5), (-1.5,-0.5)]
ys = [-1, -1, -1, -1, 1, 1, 1, 1, 1, -1]


def write_boundary_svg(points, labels, line=(1, -1, 0), path='classification_boundary.svg'):
    width = height = 360
    def sx(x): return 40 + (x + 3) / 6 * 280
    def sy(y): return 320 - (y + 3) / 6 * 280
    items = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<line x1="40" y1="180" x2="320" y2="180" stroke="#aaa"/>',
        '<line x1="180" y1="40" x2="180" y2="320" stroke="#aaa"/>',
    ]
    if line is not None:
        a,b,c = line
        x1,x2 = -3,3
        y1 = -(a*x1+c)/b if b != 0 else -3
        y2 = -(a*x2+c)/b if b != 0 else 3
        items.append(f'<line x1="{sx(x1):.1f}" y1="{sy(y1):.1f}" x2="{sx(x2):.1f}" y2="{sy(y2):.1f}" stroke="black" stroke-width="2"/>')
    for (x,y), label in zip(points, labels):
        color = '#e74c3c' if label == 1 else '#3498db'
        items.append(f'<circle cx="{sx(x):.1f}" cy="{sy(y):.1f}" r="6" fill="{color}"/>')
    items.append('</svg>')
    Path(path).write_text('\n'.join(items), encoding='utf-8')
    return path

svg_path = write_boundary_svg(xs, ys)
print('已生成:', svg_path)

## Debug Lab - 分类里最容易混的两个点

In [ ]:
# Debug 1：hinge loss 不是 accuracy。答对但 margin=0.2 时，loss 仍然大于 0。
student_loss_when_y1_score02 = None

# Debug 2：ReLU 的值不是“x>0 时为 1”。
student_relu_of_3 = None

# Debug 3：y=-1, score=-2 是答对还是答错？填字符串 'right' 或 'wrong'
student_case_negative_label = None


def qa_check_07_debug():
    values = [student_loss_when_y1_score02, student_relu_of_3, student_case_negative_label]
    if not todo_guard(values): return
    assert near(student_loss_when_y1_score02, 0.8)
    assert student_relu_of_3 == 3
    assert student_case_negative_label == 'right'
    print('OK: 分类 Debug Lab 通过。')

qa_check_07_debug()

## 提交清单

- [ ] `qa_check_07_hinge` 通过
- [ ] 生成了 `classification_boundary.svg`
- [ ] Debug Lab 通过
- [ ] 我能解释为什么要用 `y * score`

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>